<a href="https://www.kaggle.com/code/jatinnabhoya/radiology-report-rag-00-smoke-test?scriptVersionId=370141345" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install -q -U bitsandbytes

import torch, transformers
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on GPU T4")
print("torch:", torch.__version__, "| transformers:", transformers.__version__)

CUDA: True
GPU: Tesla T4
torch: 2.11.0+cu130 | transformers: 5.20.2


In [2]:
import pandas as pd, os

DATA = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university"
IMG_DIR = f"{DATA}/images/images_normalized"
reports = pd.read_csv(f"{DATA}/indiana_reports.csv")
proj = pd.read_csv(f"{DATA}/indiana_projections.csv")
print("reports:", reports.shape, "| projections:", proj.shape)

# First frontal X-ray that has a non-empty Findings section
frontal = proj[proj["projection"] == "Frontal"].merge(reports, on="uid")
row = frontal[frontal["findings"].notna()].iloc[0]
IMG = os.path.join(IMG_DIR, row["filename"])
print("Image:", IMG, "| exists:", os.path.exists(IMG))
print("Ground-truth findings:", row["findings"])

reports: (3851, 8) | projections: (7466, 3)
Image: /kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized/1_IM-0001-4001.dcm.png | exists: True
Ground-truth findings: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no XXXX of a pleural effusion. There is no evidence of pneumothorax.


In [3]:
import torch, transformers
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
print("transformers:", transformers.__version__)

MODEL = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 has no bfloat16
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(MODEL)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL, quantization_config=bnb, device_map={"": 0}, torch_dtype=torch.float16)
model.eval()
print("Loaded | GPU memory: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

transformers: 5.20.2


processor_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

... (7 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded | GPU memory: 4.3 GB


In [4]:
import time
from PIL import Image

image = Image.open(IMG).convert("RGB")
question = "You are a radiologist. Write the FINDINGS section for this chest X-ray."
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
try:
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
except Exception:
    prompt = f"[INST] <image>\n{question} [/INST]"   # Mistral format fallback
print("PROMPT:", prompt)

inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device, torch.float16)
t0 = time.time()
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
gen = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Time: %.1f s\n" % (time.time() - t0))
print("GENERATED:\n", gen)
print("\nGROUND TRUTH:\n", row["findings"])

PROMPT: [INST] <image>
You are a radiologist. Write the FINDINGS section for this chest X-ray. [/INST]
Time: 8.5 s

GENERATED:
 The chest X-ray shows several findings, including a right-sided pleural effusion, a right-sided pneumothorax, and a right-sided pneumomediastinum.

GROUND TRUTH:
 The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no XXXX of a pleural effusion. There is no evidence of pneumothorax.


In [5]:
for i in [1, 2, 3]:
    r = frontal[frontal["findings"].notna()].iloc[i]
    img = Image.open(os.path.join(IMG_DIR, r["filename"])).convert("RGB")
    inp = processor(images=img, text=prompt, return_tensors="pt").to(model.device, torch.float16)
    with torch.inference_mode():
        o = model.generate(**inp, max_new_tokens=200, do_sample=False)
    print(f"--- X-ray {i} ---")
    print("GENERATED:", processor.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    print("TRUTH:    ", r["findings"], "\n")

--- X-ray 1 ---
GENERATED: The chest X-ray shows no acute cardiopulmonary abnormalities. This means that there are no visible signs of immediate or severe issues affecting the heart or lungs in this particular image.
TRUTH:     Borderline cardiomegaly. Midline sternotomy XXXX. Enlarged pulmonary arteries. Clear lungs. Inferior XXXX XXXX XXXX. 

--- X-ray 2 ---
GENERATED: The chest X-ray shows bilateral patchy infiltrates, which are areas of increased opacity in both lungs. These infiltrates can be caused by various factors, such as infections, inflammation, or other lung conditions.
TRUTH:     There are diffuse bilateral interstitial and alveolar opacities consistent with chronic obstructive lung disease and bullous emphysema. There are irregular opacities in the left lung apex, that could represent a cavitary lesion in the left lung apex.There are streaky opacities in the right upper lobe, XXXX scarring. The cardiomediastinal silhouette is normal in size and contour. There is no pneum